# Generate S3 traffic for InstantEvidence

Use this notebook to create S3 activity in your bucket—uploads, reads, copies, listings, and deletes—so you can confirm **InstantEvidence** is receiving and evaluating events the way you expect.

Operations run through the [AWS API MCP Server](https://github.com/awslabs/mcp/tree/main/src/aws-api-mcp-server), which executes standard AWS S3 API calls using the credentials you provide.

Object keys use mixed prefixes (for example `public/`, `data/`, and `secret/`) so InstantEvidence can exercise allow/deny style rules during demos.

## Before you begin

### AWS credentials

In Google Colab, open **Secrets** (key icon in the left sidebar) and add:

| Secret | Required |
|--------|----------|
| `AWS_ACCESS_KEY_ID` | Yes |
| `AWS_SECRET_ACCESS_KEY` | Yes |
| `AWS_REGION` | No — defaults to `us-east-1` |
| `AWS_SESSION_TOKEN` | No — only if your credentials include a session token (SSO or assumed role) |

Use credentials scoped to your **test bucket only**. Minimum IAM actions:

- `sts:GetCallerIdentity`
- `s3:HeadBucket`, `s3:ListBucket` on the bucket
- `s3:PutObject`, `s3:GetObject`, `s3:HeadObject`, `s3:DeleteObject`, `s3:CopyObject` on `arn:aws:s3:::YOUR_BUCKET/*`

Avoid production admin keys.

### Your bucket and InstantEvidence

- Set **BUCKET** in the run cell to the same S3 bucket InstantEvidence monitors.
- Ensure object events from that bucket reach InstantEvidence (for example via SNS or EventBridge to your S3 event webhook).

### Helper modules

The next code cell loads `aws_credentials.py`, `mcp_s3_client.py`, and `traffic_generator.py`. It will:

1. Look beside this notebook (opening from [GitHub](https://github.com/synapse6-ai/traffic-generator) is best), or  
2. **Clone the repo automatically** into `/content/traffic-generator`, or  
3. Ask you to upload **only those three `.py` files** (not images or the notebook).

Download links: [colab folder on GitHub](https://github.com/synapse6-ai/traffic-generator/tree/main/colab).


### Run order (important in Colab)

Run cells top to bottom. The **load helpers** cell must show `Helpers loaded OK` before **Run traffic**. If MCP connection fails, use **Runtime → Restart session**, then **Run all** again.

## How to run

1. Open this notebook in [Google Colab](https://colab.research.google.com/) from GitHub (recommended) or upload the `colab/` files together.
2. Add the secrets above.
3. Set **BUCKET** to your bucket name and adjust **CYCLES** / intervals if needed.
4. Select **Runtime → Run all**.

The traffic loop usually takes a few minutes (default 20 cycles with pauses between operations). If it stops early after repeated errors, check the bucket name, secrets, and IAM permissions.



In [ ]:
%pip install -q awslabs.aws-api-mcp-server mcp nest_asyncio


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/synapse6-ai/traffic-generator.git"
MODULE_NAMES = ("aws_credentials.py", "mcp_s3_client.py", "traffic_generator.py")
CLONE_ROOT = Path("/content/traffic-generator")


def _find_module_dir() -> Path | None:
    here = Path.cwd()
    candidates: list[Path] = [
        here,
        here / "colab",
        here.parent / "colab",
        CLONE_ROOT / "colab",
        Path("/content/colab"),
        Path("/content/colab-modules"),
    ]
    for parent in [here, *here.parents]:
        if (parent / "colab" / "aws_credentials.py").exists():
            candidates.append(parent / "colab")
        if parent.name == "colab" and (parent / "aws_credentials.py").exists():
            candidates.append(parent)
    seen: set[Path] = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if all((resolved / name).exists() for name in MODULE_NAMES):
            return resolved
    return None


def _sync_repo() -> Path:
    """Clone or hard-reset to latest origin/main (avoids stale /content/traffic-generator)."""
    if CLONE_ROOT.exists() and (CLONE_ROOT / ".git").is_dir():
        print("Updating traffic-generator clone (origin/main) ...")
        subprocess.run(
            ["git", "-C", str(CLONE_ROOT), "fetch", "--depth", "1", "origin", "main"],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(CLONE_ROOT), "reset", "--hard", "origin/main"],
            check=True,
        )
    else:
        if CLONE_ROOT.exists():
            import shutil

            shutil.rmtree(CLONE_ROOT)
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                REPO_URL,
                str(CLONE_ROOT),
            ],
            check=True,
        )
    colab_dir = CLONE_ROOT / "colab"
    missing = [n for n in MODULE_NAMES if not (colab_dir / n).exists()]
    if missing:
        raise RuntimeError(f"Repo sync OK but missing under {colab_dir}: {missing}")
    mcp_src = (colab_dir / "mcp_s3_client.py").read_text()
    if "def run_notebook_async" not in mcp_src:
        raise RuntimeError(
            "mcp_s3_client.py is outdated (missing run_notebook_async). "
            "Restart runtime and re-run this cell."
        )
    return colab_dir


def _clone_repo() -> Path:
    return _sync_repo()


module_dir = _find_module_dir()
if module_dir is not None:
    mcp_file = module_dir / "mcp_s3_client.py"
    if mcp_file.exists() and "def run_notebook_async" not in mcp_file.read_text():
        print(f"Stale helpers under {module_dir}; refreshing from GitHub ...")
        module_dir = None

if module_dir is None:
    try:
        module_dir = _sync_repo()
    except Exception:
        print("Could not find helpers. Upload these three `.py` files only:")
        for name in MODULE_NAMES:
            print(f"  • {name}")
        print(
            "From: https://github.com/synapse6-ai/traffic-generator/tree/main/colab"
        )
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No files uploaded.") from None
        module_dir = Path("/content/colab-modules")
        module_dir.mkdir(parents=True, exist_ok=True)
        for name, content in uploaded.items():
            dest = module_dir / Path(name).name
            dest.write_bytes(content)
            print(f"  saved {dest.name} ({dest.stat().st_size} bytes)")
        missing = [n for n in MODULE_NAMES if not (module_dir / n).exists()]
        if missing:
            got = list(uploaded.keys())
            raise RuntimeError(
                f"Still missing {missing}. You uploaded: {got}. "
                "Upload aws_credentials.py, mcp_s3_client.py, and traffic_generator.py "
                "(not screenshots or the .ipynb)."
            )

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
print(f"Using modules from: {module_dir}")

for _name in list(sys.modules):
    if _name in ("aws_credentials", "mcp_s3_client", "traffic_generator"):
        del sys.modules[_name]

from aws_credentials import CredentialError, mask_access_key, resolve_aws_credentials
from mcp_s3_client import AwsMcpS3Client
try:
    from mcp_s3_client import run_notebook_async
except ImportError:
    import asyncio

    import nest_asyncio
    from mcp_s3_client import _real_stdio_for_mcp

    def run_notebook_async(coro):
        with _real_stdio_for_mcp():
            nest_asyncio.apply()
            return asyncio.run(coro)

from traffic_generator import run_traffic_loop, verify_access


print("Helpers loaded OK")




## Run traffic

Adjust the settings below, then run the cell. You will see one line per operation (for example `PutObject -> OK`). When the loop finishes, the cell output includes a summary of operations and any errors.

| Setting | What it does |
|---------|----------------|
| **BUCKET** | S3 bucket InstantEvidence monitors |
| **CYCLES** | How many operations to run |
| **MIN_INTERVAL_SEC** / **MAX_INTERVAL_SEC** | Random pause between operations (seconds) |
| **READ_ONLY** | When enabled, only list/read/head operations run (no uploads, copies, or deletes) |


In [ ]:
# Uses names imported in the "load helpers" cell above
from aws_credentials import CredentialError, mask_access_key, resolve_aws_credentials
from mcp_s3_client import AwsMcpS3Client
try:
    from mcp_s3_client import run_notebook_async
except ImportError:
    import asyncio

    import nest_asyncio
    from mcp_s3_client import _real_stdio_for_mcp

    def run_notebook_async(coro):
        with _real_stdio_for_mcp():
            nest_asyncio.apply()
            return asyncio.run(coro)

from traffic_generator import run_traffic_loop, verify_access


BUCKET = "your-instantevidence-bucket"  # @param {type:"string"}
CYCLES = 20  # @param {type:"integer"}
MIN_INTERVAL_SEC = 3.0  # @param {type:"number"}
MAX_INTERVAL_SEC = 12.0  # @param {type:"number"}
READ_ONLY = False  # @param {type:"boolean"}

try:
    creds = resolve_aws_credentials()
    print(f"Using {mask_access_key(creds.access_key_id)} in {creds.region}")
except CredentialError as exc:
    raise SystemExit(exc) from exc


async def run_traffic() -> object:
    mcp = AwsMcpS3Client(creds, read_only=READ_ONLY)
    try:
        await mcp.__aenter__()
        await verify_access(mcp, BUCKET)
        print("Smoke test OK")
        return await run_traffic_loop(
            bucket=BUCKET,
            credentials=creds,
            client=mcp,
            cycles=CYCLES,
            min_interval_sec=MIN_INTERVAL_SEC,
            max_interval_sec=MAX_INTERVAL_SEC,
            read_only=READ_ONLY,
            verify=False,
        )
    finally:
        await mcp.aclose()


stats = run_notebook_async(run_traffic())
stats



## View results in InstantEvidence

If your bucket is connected to InstantEvidence, you should see new S3 events in the console shortly after each operation completes. If nothing appears, confirm event delivery (SNS or EventBridge) is configured for the bucket you set in **BUCKET**.
